# 🧠 DepressoSpeech - Colab Training Notebook

**Train your trimodal depression detection model on Google Colab (free GPU)**

This notebook handles:
1. Mounting Google Drive
2. Installing dependencies
3. Building NPZ cache for fast loading
4. Training (Regression PHQ-8 or Binary Classification)
5. Saving checkpoints to Drive

**Before you start:**
- Upload your `Model/` folder to Google Drive (e.g., `MyDrive/DepressoSpeech/Model/`)
- Ensure your data is in `Model/data/raw/<PID>/` with all 6 feature files
- Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`

## 1. 🔌 Mount Google Drive

Run this cell and authorize access to your Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
!ls -la /content/drive/MyDrive/

## 2. 📁 Set Project Path

**EDIT THIS:** Change to your Model folder location in Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURE THIS: Path to your Model folder in Google Drive
# ═══════════════════════════════════════════════════════════════════════════

PROJECT_PATH = '/content/drive/MyDrive/DepressoSpeech/Model'  # ← EDIT THIS

# Verify path exists
import os
if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(f"Project not found at: {PROJECT_PATH}\n"
                            f"Please update PROJECT_PATH to your Model folder location")

print(f"✓ Project path: {PROJECT_PATH}")
!ls -la {PROJECT_PATH}

## 3. 📦 Install Dependencies

Installs PyTorch, OpenSMILE, librosa, sentence-transformers, etc.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Install required packages
# ═══════════════════════════════════════════════════════════════════════════

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opensmile librosa soundfile
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q scikit-learn pandas numpy pyyaml tqdm

# Verify installations
import torch
print(f"✓ PyTorch {torch.__version__}")
print(f"✓ CUDA Available: {torch.cuda.is_available()}")
print(f"✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 4. 🏗️ Setup Project & Imports

In [ ]:
import sys
import os
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import logging
from tqdm import tqdm

# Add project to path
sys.path.insert(0, PROJECT_PATH)

# Change to project directory
os.chdir(PROJECT_PATH)

# Project imports
from src.models.multimodal_fusion_v2 import TrimodalFusionModel, FusionOutput
from src.dataset.trimodal_dataset import (
    TrimodalDataset, trimodal_collate_fn,
    build_participant_data, build_npz_cache
)
from src.training.losses import WeightedMSELoss, CombinedLoss, WeightedBCELoss
from src.training.metrics import compute_all_metrics, ClassificationMetricsTracker
from src.training.early_stopping import EarlyStopping

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")
print(f"✓ Project root: {PROJECT_PATH}")

## 5. ⚙️ Training Configuration

**EDIT THESE SETTINGS** for your training run

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING CONFIGURATION - EDIT THESE VALUES
# ═══════════════════════════════════════════════════════════════════════════

# --- Task Selection ---
TASK = "classification"  # Options: "regression" (PHQ-8 score) or "classification" (depressed/not)

# --- Model Architecture ---
FUSION_DIM = 128           # Shared projection dimension
NUM_ATTENTION_HEADS = 4   # Cross-modal attention heads
DROPOUT = 0.3             # Dropout rate (0.2-0.4 recommended)
AUDIO_TEMPORAL = "conv"   # "conv" (default, fewer params) or "gru" (long-range, more params)

# --- Data Paths (relative to PROJECT_PATH) ---
RAW_DATA_DIR = "data/raw"           # Where participant folders are
LABELS_CSV = "data/labels.csv"      # PHQ-8 scores
NPZ_CACHE_DIR = "data/npz"          # NPZ cache for fast loading
CHECKPOINT_DIR = "checkpoints"       # Where to save models

# --- Training Hyperparameters ---
BATCH_SIZE = 16           # Reduce to 8 if OOM (T4 has 16GB)
LEARNING_RATE = 1e-4      # Classification: 1e-4, Regression: 5e-3 (stage1) → 1e-3 (stage3)
WEIGHT_DECAY = 1e-3       # L2 regularization
EPOCHS = 200              # Max epochs (early stopping will cut this short)
PATIENCE = 20             # Early stopping patience
GRADIENT_CLIP = 1.0       # Gradient clipping norm

# --- Data Split ---
TEST_SIZE = 0.15          # Fraction for test set
VAL_SIZE = 0.15           # Fraction for validation (from remaining)
RANDOM_SEED = 42          # For reproducibility

# --- Classification Specific ---
PHQ_THRESHOLD = 10.0      # PHQ-8 ≥ 10 = depressed (for classification)
FOCAL_GAMMA = 0.0         # 0 = no focal loss, 2.0 = strong focal
LABEL_SMOOTHING = 0.1     # Label smoothing for BCE

# --- Augmentation ---
USE_AUGMENTATION = True
TEMPORAL_DROPOUT = 0.15   # Frame dropout rate
FEATURE_NOISE = 0.008     # Gaussian noise std
MODALITY_DROPOUT = 0.15   # Probability to drop entire modality

# Print config
print("Training Configuration:")
print(f"  Task: {TASK}")
print(f"  Audio Temporal: {AUDIO_TEMPORAL}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS} (with early stopping patience={PATIENCE})")

## 6. 📊 Load & Prepare Data

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Load Labels and Build/Check NPZ Cache
# ═══════════════════════════════════════════════════════════════════════════

labels_path = Path(PROJECT_PATH) / LABELS_CSV
raw_path = Path(PROJECT_PATH) / RAW_DATA_DIR
npz_path = Path(PROJECT_PATH) / NPZ_CACHE_DIR
checkpoint_path = Path(PROJECT_PATH) / CHECKPOINT_DIR

# Load labels
labels_df = pd.read_csv(labels_path)
print(f"✓ Loaded {len(labels_df)} labels")
print(f"  Columns: {list(labels_df.columns)}")

# Check if NPZ cache exists, build if not
if not npz_path.exists() or not any(npz_path.glob("*.npz")):
    print(f"\n⚠ NPZ cache not found at {npz_path}")
    print("Building NPZ cache from raw CSVs (one-time operation)...")
    
    # Create cache directory
    npz_path.mkdir(parents=True, exist_ok=True)
    
    # Build cache
    build_npz_cache(
        raw_dir=str(raw_path),
        output_dir=str(npz_path),
        labels_df=labels_df,
        id_col="Participant_ID"
    )
    print(f"✓ NPZ cache built at {npz_path}")
else:
    print(f"✓ NPZ cache found: {len(list(npz_path.glob('*.npz')))} files")

# Build participant data (loads from NPZ if available)
print("\nLoading participant features...")
participant_data = build_participant_data(
    raw_dir=str(raw_path),
    labels_df=labels_df,
    id_col="Participant_ID",
    npz_cache_dir=str(npz_path) if npz_path.exists() else None
)

print(f"✓ Loaded {len(participant_data)} participants")
if participant_data:
    sample_pid = list(participant_data.keys())[0]
    sample_data = participant_data[sample_pid]
    print(f"\nSample participant {sample_pid}:")
    for key, value in sample_data.items():
        if value is not None:
            print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
        else:
            print(f"  {key}: None (missing)")

## 7. ✂️ Create Train/Validation/Test Splits

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Create Stratified Splits
# ═══════════════════════════════════════════════════════════════════════════

# Get all participant IDs with features
all_pids = list(participant_data.keys())
print(f"Total participants with features: {len(all_pids)}")

# Create labels dictionary
labels_dict = {}
for _, row in labels_df.iterrows():
    pid = str(int(row["Participant_ID"]))
    if pid in participant_data:
        labels_dict[pid] = float(row["PHQ_Score"])

# For stratification, use PHQ-8 bins
phq_scores = [labels_dict[pid] for pid in all_pids]
phq_bins = [int(score // 5) for score in phq_scores]  # Bin by severity

# First split: separate test set
train_val_pids, test_pids = train_test_split(
    all_pids, 
    test_size=TEST_SIZE, 
    stratify=phq_bins,
    random_state=RANDOM_SEED
)

# Second split: separate train and validation
train_pids, val_pids = train_test_split(
    train_val_pids,
    test_size=VAL_SIZE / (1 - TEST_SIZE),  # Adjust for remaining data
    stratify=[labels_dict[pid] // 5 for pid in train_val_pids],
    random_state=RANDOM_SEED
)

print(f"\nData Splits:")
print(f"  Train: {len(train_pids)} participants")
print(f"  Validation: {len(val_pids)} participants")
print(f"  Test: {len(test_pids)} participants")

# Show class distribution for classification
if TASK == "classification":
    for split_name, split_pids in [("Train", train_pids), ("Val", val_pids), ("Test", test_pids)]:
        depressed = sum(1 for pid in split_pids if labels_dict[pid] >= PHQ_THRESHOLD)
        not_depressed = len(split_pids) - depressed
        print(f"  {split_name}: {depressed} depressed, {not_depressed} not-depressed")

## 8. 🔄 Create DataLoaders

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Prepare Labels and Create DataLoaders
# ═══════════════════════════════════════════════════════════════════════════

# Prepare labels based on task
if TASK == "classification":
    # Binary labels: 1 if PHQ-8 >= threshold, else 0
    task_labels = {pid: 1.0 if labels_dict[pid] >= PHQ_THRESHOLD else 0.0 for pid in all_pids}
    print(f"Binary classification: PHQ-8 ≥ {PHQ_THRESHOLD} = depressed")
else:
    # Regression: use raw PHQ-8 scores
    task_labels = labels_dict
    print("Regression: Predicting PHQ-8 scores (0-24)")

# Create datasets
train_dataset = TrimodalDataset(
    participant_data=participant_data,
    labels=task_labels,
    participant_ids=train_pids,
    augment=USE_AUGMENTATION,
    temporal_dropout_rate=TEMPORAL_DROPOUT,
    feature_noise_std=FEATURE_NOISE,
    label_noise_std=0.2 if TASK == "regression" else 0.0,
    depression_threshold=PHQ_THRESHOLD,
    modality_dropout_prob=MODALITY_DROPOUT,
)

val_dataset = TrimodalDataset(
    participant_data=participant_data,
    labels=task_labels,
    participant_ids=val_pids,
    augment=False,
)

test_dataset = TrimodalDataset(
    participant_data=participant_data,
    labels=task_labels,
    participant_ids=test_pids,
    augment=False,
)

# Create DataLoaders (num_workers=0 for Colab compatibility)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=trimodal_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=trimodal_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=trimodal_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False,
)

print(f"\n✓ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 9. 🏗️ Initialize Model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Initialize TrimodalFusionModel
# ═══════════════════════════════════════════════════════════════════════════

# Model configuration
num_classes = 2 if TASK == "classification" else 1

# Feature dimensions from config
openface_dim = 49
cnn_dim = 512

model = TrimodalFusionModel(
    fusion_dim=FUSION_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    stats_mode="mean_std",
    dropout=DROPOUT,
    modality_dropout=MODALITY_DROPOUT,
    openface_dim=openface_dim,
    cnn_dim=cnn_dim,
    use_spectral_norm=False,
    stochastic_depth_rate=0.1,
    attention_dropout=DROPOUT / 2,
    audio_temporal=AUDIO_TEMPORAL,
    num_classes=num_classes,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model initialized:")
print(f"  Task: {TASK} (num_classes={num_classes})")
print(f"  Audio temporal: {AUDIO_TEMPORAL}")
print(f"  Total params: {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

## 10. 🎯 Setup Loss, Optimizer, Scheduler

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Setup Training Components
# ═══════════════════════════════════════════════════════════════════════════

# Loss function
if TASK == "classification":
    # Compute pos_weight from class imbalance
    train_labels = [task_labels[pid] for pid in train_pids]
    n_positive = sum(train_labels)
    n_negative = len(train_labels) - n_positive
    pos_weight = n_negative / max(n_positive, 1)
    
    criterion = WeightedBCELoss(
        pos_weight=pos_weight,
        focal_gamma=FOCAL_GAMMA,
        label_smoothing=LABEL_SMOOTHING,
    ).to(device)
    
    # Metrics tracker
    metrics_tracker = ClassificationMetricsTracker()
    
    print(f"✓ Classification loss (WeightedBCE)")
    print(f"  pos_weight: {pos_weight:.2f} (from {n_positive:.0f}+ / {n_negative:.0f}-)")
    print(f"  focal_gamma: {FOCAL_GAMMA}")
    print(f"  label_smoothing: {LABEL_SMOOTHING}")
    
else:
    # Regression loss
    criterion = CombinedLoss(
        phq_threshold=PHQ_THRESHOLD,
        high_weight=2.0,
        low_weight=1.0,
        ccc_weight=0.5,
    ).to(device)
    
    print(f"✓ Regression loss (Combined: WeightedMSE + CCC)")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max' if TASK == "classification" else 'max',  # Maximize F1 or CCC
    factor=0.5,
    patience=PATIENCE // 2,
    verbose=True
)

# Early stopping
early_stop = EarlyStopping(
    patience=PATIENCE,
    min_delta=0.001,
    mode='max',
)

# Ensure checkpoint directory exists
checkpoint_path.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"✓ Scheduler: ReduceLROnPlateau (patience={PATIENCE//2})")
print(f"✓ Early stopping: patience={PATIENCE}")
print(f"✓ Checkpoints: {checkpoint_path}")

## 11. 🏋️ Training Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Training and Validation Functions
# ═══════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, task_type):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    n_batches = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        # Prepare inputs
        kwargs = {}
        if batch['has_audio'].any():
            kwargs['mfcc'] = batch['audio_features'][..., :120].to(device)
            kwargs['egemaps'] = batch['audio_features'][..., 120:].to(device)
            kwargs['audio_mask'] = batch['audio_mask'].to(device)
        
        if batch['has_video'].any():
            kwargs['openface'] = batch['video_openface'].to(device)
            kwargs['cnn_embed'] = batch['video_cnn'].to(device)
            kwargs['video_mask'] = batch['video_mask'].to(device)
        
        if batch['has_text'].any():
            kwargs['text'] = batch['text_features'].to(device)
            kwargs['text_mask'] = batch['text_mask'].to(device)
        
        labels = batch['labels'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(**kwargs)
        
        # Compute loss
        if task_type == "classification":
            # Binary classification: BCE with logits
            preds = output.prediction.squeeze()
            loss = criterion(preds, labels)
        else:
            # Regression: clamp predictions to [0, 24]
            preds = output.prediction.squeeze()
            loss = criterion(preds, labels)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def validate(model, loader, criterion, task_type, metrics_tracker=None):
    """Validate and compute metrics."""
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0
    n_batches = 0
    
    for batch in tqdm(loader, desc="Validation"):
        kwargs = {}
        if batch['has_audio'].any():
            kwargs['mfcc'] = batch['audio_features'][..., :120].to(device)
            kwargs['egemaps'] = batch['audio_features'][..., 120:].to(device)
            kwargs['audio_mask'] = batch['audio_mask'].to(device)
        
        if batch['has_video'].any():
            kwargs['openface'] = batch['video_openface'].to(device)
            kwargs['cnn_embed'] = batch['video_cnn'].to(device)
            kwargs['video_mask'] = batch['video_mask'].to(device)
        
        if batch['has_text'].any():
            kwargs['text'] = batch['text_features'].to(device)
            kwargs['text_mask'] = batch['text_mask'].to(device)
        
        labels = batch['labels'].to(device)
        
        output = model(**kwargs)
        preds = output.prediction.squeeze()
        loss = criterion(preds, labels)
        
        if task_type == "regression":
            preds = preds.clamp(0, 24)
        
        all_preds.append(preds.cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        total_loss += loss.item()
        n_batches += 1
    
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / max(n_batches, 1)
    
    if task_type == "classification" and metrics_tracker is not None:
        metrics = metrics_tracker.update(all_preds, all_targets)
        metrics['loss'] = avg_loss
    else:
        metrics = compute_all_metrics(all_preds, all_targets)
        metrics['loss'] = avg_loss
    
    return metrics

print("✓ Training functions defined")

## 12. 🚀 Training Loop

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Main Training Loop
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print(f"STARTING {TASK.upper()} TRAINING")
print(f"{'='*60}")

best_metric = 0.0
best_epoch = 0
history = []

# Determine metric name based on task
metric_name = "f1" if TASK == "classification" else "ccc"

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 60)
    
    # Train
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, TASK)
    
    # Validate
    val_metrics = validate(model, val_loader, criterion, TASK, metrics_tracker)
    
    # Get current metric
    current_metric = val_metrics.get(metric_name, 0.0)
    
    # Scheduler step
    scheduler.step(current_metric)
    
    # Print metrics
    if TASK == "classification":
        print(f"Train Loss: {train_loss:.4f} | "
              f"Val Acc: {val_metrics['accuracy']:.3f} | "
              f"F1: {val_metrics['f1']:.3f} | "
              f"ROC-AUC: {val_metrics['roc_auc']:.3f}")
    else:
        print(f"Train Loss: {train_loss:.4f} | "
              f"Val CCC: {val_metrics['ccc']:.4f} | "
              f"RMSE: {val_metrics['rmse']:.3f} | "
              f"MAE: {val_metrics['mae']:.3f}")
    
    # Save best model
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        
        checkpoint_file = checkpoint_path / f"best_{TASK}.pt"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'config': {
                'task': TASK,
                'fusion_dim': FUSION_DIM,
                'num_attention_heads': NUM_ATTENTION_HEADS,
                'dropout': DROPOUT,
                'audio_temporal': AUDIO_TEMPORAL,
            }
        }, checkpoint_file)
        print(f"✓ Saved best model (epoch {epoch}, {metric_name}={current_metric:.4f})")
    
    # Early stopping check
    if early_stop(current_metric, epoch):
        print(f"\n⚠ Early stopping triggered at epoch {epoch}")
        break

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best {metric_name.upper()}: {best_metric:.4f} at epoch {best_epoch}")
print(f"Checkpoint: {checkpoint_path}/best_{TASK}.pt")
print(f"{'='*60}")

## 13. 📊 Final Evaluation on Test Set

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Evaluate on Test Set
# ═══════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*60}")

# Load best model
checkpoint_file = checkpoint_path / f"best_{TASK}.pt"
if checkpoint_file.exists():
    checkpoint = torch.load(checkpoint_file)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
else:
    print("⚠ Using current model (no checkpoint found)")

# Evaluate
test_metrics = validate(model, test_loader, criterion, TASK, metrics_tracker)

print(f"\nTest Set Results:")
if TASK == "classification":
    print(f"  Accuracy:  {test_metrics['accuracy']:.3f}")
    print(f"  F1 Score:  {test_metrics['f1']:.3f}")
    print(f"  ROC-AUC:   {test_metrics['roc_auc']:.3f}")
    print(f"  Precision: {test_metrics['precision']:.3f}")
    print(f"  Recall:    {test_metrics['recall']:.3f}")
    print(f"\nConfusion Matrix:")
    print(f"  [[TN FP]")
    print(f"   [FN FP]] = {test_metrics['confusion_matrix']}")
else:
    print(f"  CCC:  {test_metrics['ccc']:.4f}")
    print(f"  RMSE: {test_metrics['rmse']:.3f}")
    print(f"  MAE:  {test_metrics['mae']:.3f}")

# Save results to file
results_file = checkpoint_path / f"{TASK}_test_results.txt"
with open(results_file, 'w') as f:
    f.write(f"Task: {TASK}\n")
    f.write(f"Best Epoch: {best_epoch}\n")
    f.write(f"\nTest Metrics:\n")
    for key, value in test_metrics.items():
        if isinstance(value, (int, float)):
            f.write(f"  {key}: {value:.4f}\n")
    f.write(f"\nConfiguration:\n")
    f.write(f"  fusion_dim: {FUSION_DIM}\n")
    f.write(f"  num_attention_heads: {NUM_ATTENTION_HEADS}\n")
    f.write(f"  dropout: {DROPOUT}\n")
    f.write(f"  audio_temporal: {AUDIO_TEMPORAL}\n")
    f.write(f"  batch_size: {BATCH_SIZE}\n")
    f.write(f"  learning_rate: {LEARNING_RATE}\n")

print(f"\n✓ Results saved to: {results_file}")

## 14. 💾 Download Checkpoints (Optional)

Run this to download your trained model to local machine

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Download Checkpoint
# ═══════════════════════════════════════════════════════════════════════════

from google.colab import files

checkpoint_file = checkpoint_path / f"best_{TASK}.pt"
if checkpoint_file.exists():
    files.download(str(checkpoint_file))
    print(f"✓ Downloaded {checkpoint_file.name}")
else:
    print("⚠ Checkpoint not found")

## 15. 🔄 Resume Training (Optional)

If training was interrupted, run this cell to resume from the last checkpoint

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Resume from Checkpoint
# ═══════════════════════════════════════════════════════════════════════════

# Uncomment and modify to resume training

# RESUME_CHECKPOINT = checkpoint_path / "best_classification.pt"  # Change this
# START_EPOCH = 50  # Epoch to resume from

# if RESUME_CHECKPOINT.exists():
#     checkpoint = torch.load(RESUME_CHECKPOINT)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     print(f"✓ Resumed from {RESUME_CHECKPOINT} (epoch {checkpoint['epoch']})")
#     
#     # Continue training from START_EPOCH
#     for epoch in range(START_EPOCH, EPOCHS + 1):
#         # ... (copy training loop code here)
# else:
#     print("⚠ Checkpoint not found")

print("To resume training, uncomment and configure the code above")